# B4 — Table B.3 (계산비용: 학습·추론 시간)

**생성물**: 부록 Table B.3의 학습시간(시드당)·슬롯당 추론시간. (R2 Comment 8)

- **학습시간 = 차분법**: 짧은 학습 2회(N1=100, N2=300 업데이트)의 시간 차로 per-update 시간을 구하고
  (워밍업·셋업·최종평가 고정비 상쇄), 논문 설정 업데이트 수(50,000 / 10,000)로 외삽.
- **추론시간**: 학습된 정책의 슬롯당 forward-pass (CPU 단일 스레드, 3,000회 평균).
- MPC ~690 ms/slot은 MPC 실행 로그 기준 (여기서 재측정하지 않음).
- 아무것도 저장하지 않음 (체크포인트·json 무변경). 실행 시간: GPU에서 약 5–10분.
- 참고: 원고 게재값(14.5/57.3/65.8/3.5분)은 동일 하드웨어(RTX 5060 Laptop)에서 측정 — 다른 기기에선 비율만 유사.

In [1]:
import os, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REV  = os.path.abspath(os.path.join(os.getcwd(), ".."))  # repo root
CODE = os.path.join(REV, "code"); sys.path.insert(0, CODE)
RES  = os.path.join(REV, "results")
ANA  = os.path.join(REV, "analysis")


In [2]:
# 학습시간 (차분법) — 최종 구현(coop_dasac, coop_dabc) 기준
import time
from data.settings import Data_Train, Data_Val, Data_Test, Data_Expert_Train, Parameters
from envs.env_multi import DualAgentEnv
from envs.env_single import SingleAgentEnv

TOL, DEG = 5.0, 5.0
BIG = 10**9              # 주기적 평가 비활성화
N1, N2 = 100, 300

def dual(data): return DualAgentEnv(data, Parameters, tol=TOL, degradation_cost_per_mwh=DEG)
def sing(data): return SingleAgentEnv(data, Parameters, degradation_cost_per_mwh=DEG)

def measure_sac(train_fn, builder, n_full, label):
    etr, eva, ete = builder(Data_Train), builder(Data_Val), builder(Data_Test)
    def run(nup):
        t0 = time.perf_counter()
        train_fn(etr, eva, ete, seed=0, start_steps=600, update_limit=nup,
                 eval_freq=BIG, save_actor_path=None)
        return time.perf_counter() - t0
    t1, t2 = run(N1), run(N2)
    per = (t2 - t1) / (N2 - N1); tot = per * n_full
    print(f"{label:<12} per-update={per*1000:7.2f} ms  x{n_full:,} = {tot/60:6.1f} min")
    return per, tot

def measure_bc(n_full, label):
    from algorithms.coop_dabc import train as bc_train
    from envs.run_expert_buffer import populate_expert_buffers
    etr, eva, ete = dual(Data_Train), dual(Data_Val), dual(Data_Test)
    bb, bo, _, _ = populate_expert_buffers(Data_Train, Data_Expert_Train, Parameters,
                                           tol=TOL, degradation_cost_per_mwh=DEG)
    def run(nst):
        t0 = time.perf_counter()
        bc_train(etr, eva, ete, bb, bo, seed=0, max_steps=nst,
                 eval_freq=BIG, save_actor_path=None)
        return time.perf_counter() - t0
    t1, t2 = run(N1), run(N2)
    per = (t2 - t1) / (N2 - N1); tot = per * n_full
    print(f"{label:<12} per-update={per*1000:7.2f} ms  x{n_full:,} = {tot/60:6.1f} min")
    return per, tot

from algorithms.single_sac import train as t_single
from algorithms.inde_sac import train as t_inde
from algorithms.coop_dasac import train as t_coop
train_res = {}
train_res["Single-SAC"] = measure_sac(t_single, sing, 50_000, "Single-SAC")
train_res["Inde-DASAC"] = measure_sac(t_inde, dual, 50_000, "Inde-DASAC")
train_res["Coop-DASAC"] = measure_sac(t_coop, dual, 50_000, "Coop-DASAC")
train_res["Coop-DABC"]  = measure_bc(10_000, "Coop-DABC")
print("\nMPC: no training (optimization at inference).")


[ 100/100  100%  7s] val=72502  best=72502


[ 300/300  100%  12s] val=73273  best=73273


Single-SAC   per-update=  16.73 ms  x50,000 =   13.9 min


[ 100/100  100%  27s] val=69250  best=69250


[ 300/300  100%  38s] val=73381  best=73381


...

[ 100/100  100%  27s] val=73315  best=73315


[ 300/300  100%  45s] val=91429  best=91429


Coop-DASAC   per-update=  93.76 ms  x50,000 =   78.1 min
[Expert buffer] 136 episodes  tol=5.0MW  deg=$5.0/MWh
[Expert buffer] total_return=692125  buf_bid.size=13056  buf_ope.size=13056
[Coop-DABC-Mono] train=136d  steps=100  alpha=2.5


[Coop-DABC-Mono] train=136d  steps=300  alpha=2.5


Coop-DABC    per-update=  24.13 ms  x10,000 =    4.0 min

MPC: no training (optimization at inference).


In [3]:
# 추론시간 (CPU 단일 스레드, 슬롯당 forward-pass)
import time, torch
from models.networks import Actor, SquashedGaussianMLPActor, SingleAgentActor

torch.set_num_threads(1)
H, N_IT, WARM = 128, 3000, 300
denv = dual(Data_Test); senv = sing(Data_Test)
obd, opd, adim = denv.observation_dim_bid(), denv.observation_dim_ope(), denv.action_dim()
lb, lo = denv.action_limit_bid(), denv.action_limit_ope()
sod, slim = senv.observation_dim_bid(), senv.action_limit()
xb, xo, xs = torch.randn(1, obd), torch.randn(1, opd), torch.randn(1, sod)

def load(net, folder, fname, key=None):
    p = os.path.join(RES, folder, fname)
    if os.path.exists(p):
        sd = torch.load(p, map_location="cpu")
        net.load_state_dict(sd[key] if key else sd)
    else:
        print(f"  [warn] missing {fname} -> random init (timing unaffected)")
    return net.eval()

a_bc  = load(Actor(obd, opd, adim, H, lb, lo), "coop_dabc", "actor_coop_dabc_deg5.0_tol5.0_seed0.pth")
a_sac = load(SquashedGaussianMLPActor(obd, opd, adim, H, lb, lo), "coop_dasac", "actor_coop_dasac_deg5.0_tol5.0_seed0.pth")
a_sng = load(SingleAgentActor(sod, adim, H, slim), "baseline", "actor_single_sac_deg5.0_tol5.0_seed0.pth")
i_bid = load(SingleAgentActor(obd, adim, H, lb), "baseline", "actor_inde_sac_deg5.0_tol5.0_seed0.pth", key="bid")
i_ope = load(SingleAgentActor(opd, adim, H, lo), "baseline", "actor_inde_sac_deg5.0_tol5.0_seed0.pth", key="ope")

def bench(name, fn):
    with torch.no_grad():
        for _ in range(WARM): fn()
        t0 = time.perf_counter()
        for _ in range(N_IT): fn()
        dt = (time.perf_counter() - t0) / N_IT * 1000.0
    print(f"{name:<12} {dt:8.4f} ms/step")
    return dt

infer = {}
infer["Single-SAC"] = bench("Single-SAC", lambda: a_sng(xs, deterministic=True, with_logprob=False))
infer["Inde-DASAC"] = bench("Inde-DASAC", lambda: (i_bid(xb, deterministic=True, with_logprob=False),
                                                   i_ope(xo, deterministic=True, with_logprob=False)))
infer["Coop-DASAC"] = bench("Coop-DASAC", lambda: a_sac(xb, xo, deterministic=True, with_logprob=False))
infer["Coop-DABC"]  = bench("Coop-DABC",  lambda: a_bc(xb, xo))
print(f"\nMPC          ~690      ms/step  (2 MILP solves, from logs)")
print(f"Speedup vs MPC: ~{690/max(infer.values()):,.0f}x - {690/min(infer.values()):,.0f}x")


Single-SAC     0.1324 ms/step


Inde-DASAC     0.2594 ms/step


Coop-DASAC     0.3957 ms/step


Coop-DABC      0.2396 ms/step

MPC          ~690      ms/step  (2 MILP solves, from logs)
Speedup vs MPC: ~1,744x - 5,213x


In [4]:
# Table B.3 형태로 요약
rows = [[k, f"{train_res[k][1]/60:.1f} min", f"{infer[k]:.2f} ms"] for k in train_res]
rows.append(["MPC", "none", "~690 ms"])
pd.DataFrame(rows, columns=["Method", "Training (per seed)", "Inference (per slot)"])


,Method,Training (per seed),Inference (per slot)
0,Single-SAC,13.9 min,0.13 ms
1,Inde-DASAC,43.2 min,0.26 ms
2,Coop-DASAC,78.1 min,0.40 ms
3,Coop-DABC,4.0 min,0.24 ms
4,MPC,none,~690 ms
